# 01 — Data Download & Preprocessing

This notebook downloads **MBPP** (via HuggingFace `datasets`) and **SPIDER** (from Yale NLP GitHub)(https://yale-lily.github.io/spider),
formats both into a unified JSONL format with task-prefix prompts, and saves them to `data/processed/`.

**Run this notebook once** before training. All subsequent notebooks depend on the JSONL files produced here.

---
| Dataset | Task | Prompt Prefix | Split sizes |
|---------|------|--------------|-------------|
| MBPP (sanitized) | Python code generation | `Generate Python: ...` | ~374 train / 100 test |
| SPIDER | Text-to-SQL | `Translate to SQL: ...` | ~7000 train / 1,659 test |

In [1]:
!pip install -q datasets transformers sentencepiece tqdm
print("✓ Dependencies installed")

✓ Dependencies installed


In [2]:
# =============================================================================
# CELL 2 — Configuration & Google Drive Setup
# =============================================================================

import os
import sys
import json
import zipfile
import shutil
import urllib.request
from pathlib import Path

# ── Mount Google Drive so files persist across Colab sessions ─────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Set your project folder on Google Drive ───────────────────────────────────
# This is where all data and model files will be saved.
# Change "FlanT5_Finetune" to whatever folder name you prefer.
PROJECT_ROOT = Path("/content/drive/MyDrive/FlanT5_Finetune")

# ── Data paths (no need to change these) ─────────────────────────────────────
RAW_MBPP_DIR    = PROJECT_ROOT / "data" / "raw"   / "mbpp"
RAW_SPIDER_DIR  = PROJECT_ROOT / "data" / "raw"   / "spider"
PROCESSED_DIR   = PROJECT_ROOT / "data" / "processed"

MBPP_TRAIN_OUT  = PROCESSED_DIR / "mbpp_train.jsonl"
MBPP_TEST_OUT   = PROCESSED_DIR / "mbpp_test.jsonl"
SPIDER_TRAIN_OUT = PROCESSED_DIR / "spider_train.jsonl"
SPIDER_TEST_OUT  = PROCESSED_DIR / "spider_test.jsonl"

# Model name — used for tokenizer in statistics cell
MODEL_NAME = "google/flan-t5-base"

# ── Create all required directories ──────────────────────────────────────────
for d in [RAW_MBPP_DIR, RAW_SPIDER_DIR, PROCESSED_DIR,
          PROJECT_ROOT / "results" / "base_model",
          PROJECT_ROOT / "results" / "finetuned"]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ Directories  : ready")

Mounted at /content/drive
✓ Project root : /content/drive/MyDrive/FlanT5_Finetune
✓ Directories  : ready


In [3]:
# CELL 3 — Helper Functions
# =============================================================================

# ── Prompt builder for MBPP ───────────────────────────────────────────────────
def build_mbpp_prompt(example):
    """
    Format an MBPP example as a task-prefix input string.
    Full MBPP dataset uses 'text' as the task description field.

    Input:  "Generate Python: {task description}\nTest cases:\n{assert statements}"
    Output: "{python code}"
    """
    task_desc  = example["text"].strip()
    test_cases = "\n".join(example.get("test_list", []))
    return (
        f"Generate Python: {task_desc}\n"
        f"Test cases:\n{test_cases}"
    )

# ── Prompt builder for SPIDER ─────────────────────────────────────────────────
def build_spider_prompt(example, schema_lookup):
    """
    Format a SPIDER example as a schema-enriched task-prefix input string.

    Input:  "Translate to SQL: {question}\nDatabase schema:\n{compact schema}"
    Output: "{SQL query}"

    Including the schema is critical — SPIDER column/table names are often
    not inferable from the question alone (e.g. LifeExpectancy, IsOfficial).
    """
    question = example["question"].strip()
    schema   = schema_lookup.get(example["db_id"], "")
    return (
        f"Translate to SQL: {question}\n"
        f"Database schema:\n{schema}"
    )

# ── Schema loader for SPIDER ──────────────────────────────────────────────────
def load_spider_schemas(tables_json_path):
    """
    Parse SPIDER's tables.json and return a dict:
        { db_id -> "table1: col1 (type), col2 (type) | table2: ..." }

    This compact linear format is standard in Text-to-SQL literature and
    fits within the 512-token input limit for most examples.
    """
    with open(tables_json_path, "r", encoding="utf-8") as f:
        tables = json.load(f)

    schemas = {}
    for db in tables:
        db_id        = db["db_id"]
        table_names  = db["table_names_original"]
        column_names = db["column_names_original"]   # [[table_idx, col_name], ...]
        column_types = db["column_types"]

        # Group columns under their parent table
        table_cols = {i: [] for i in range(len(table_names))}
        for (tbl_idx, col_name), col_type in zip(column_names, column_types):
            if tbl_idx == -1:   # skip the special "*" wildcard column
                continue
            table_cols[tbl_idx].append(f"{col_name} ({col_type})")

        parts = [
            f"{name}: {', '.join(table_cols[i])}"
            for i, name in enumerate(table_names)
        ]
        schemas[db_id] = " | ".join(parts)

    return schemas

# ── JSONL writer ──────────────────────────────────────────────────────────────
def write_jsonl(records, path):
    """Write a list of dicts to a JSONL file (one JSON object per line)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"  ✓ Saved {len(records):,} records → {path}")

print("✓ Helper functions defined")

✓ Helper functions defined


In [4]:
# =============================================================================
# CELL 4 — MBPP: Download & Preprocess
# =============================================================================
# Uses the FULL MBPP dataset (not sanitized) for more training data.
# Prompt format:
#   Input : "Generate Python: {task description}\nTest cases:\n{asserts}"
#   Output: "{python code}"
#
# Full dataset split sizes:
#   train: 374 | validation: 90 | test: 500 | prompt: 10
#
# We merge train + validation → 464 training examples.
# We use test (500 examples) as the held-out evaluation set.
# Note: full dataset uses field 'text' (not 'prompt' like the sanitized split).

from datasets import load_dataset

def normalize_code(code: str) -> str:
    """
    Normalize raw Python code strings from MBPP:
      1. Convert Windows-style line endings (\r\n) to Unix (\n)
      2. Expand tabs to 4 spaces (consistent indentation)
      3. Strip trailing whitespace from every line
    """
    code = code.replace("\r\n", "\n")                       # 1. CRLF → LF
    code = code.expandtabs(tabsize=4)                        # 2. tabs → 4 spaces
    lines = [line.rstrip() for line in code.split("\n")]    # 3. trailing whitespace
    return "\n".join(lines).strip()                          # final strip of surrounding blank lines


print("Downloading full MBPP dataset...")
mbpp_dataset = load_dataset("google-research-datasets/mbpp")   # full, not sanitized
print(mbpp_dataset)

def process_mbpp_split(split):
    """Convert a HuggingFace MBPP split into a list of input/output dicts."""
    records = []
    for ex in split:
        records.append({
            "input":     build_mbpp_prompt(ex),
            "output":    normalize_code(ex["code"]),         # normalized: CRLF, tabs, trailing ws
            "task_id":   ex.get("task_id", ""),
            "test_list": ex.get("test_list", []),            # kept for pass@1 evaluation
        })
    return records

# Merge train + validation to maximise training examples
mbpp_train = (
    process_mbpp_split(mbpp_dataset["train"]) +
    process_mbpp_split(mbpp_dataset["validation"])
)
mbpp_test = process_mbpp_split(mbpp_dataset["test"])

print(f"\nMBPP → {len(mbpp_train):,} train (train+validation)  |  {len(mbpp_test):,} test")
print("\nSample input prompt:")
print(mbpp_train[0]["input"])
print("\nSample target (Python code):")
print(mbpp_train[0]["output"])

write_jsonl(mbpp_train, MBPP_TRAIN_OUT)
write_jsonl(mbpp_test,  MBPP_TEST_OUT)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/87.2k [00:00<?, ?B/s]

full/test-00000-of-00001.parquet:   0%|          | 0.00/116k [00:00<?, ?B/s]

full/validation-00000-of-00001.parquet:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 374
    })
    test: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 90
    })
    prompt: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 10
    })
})

MBPP → 464 train (train+validation)  |  500 test

Sample input prompt:
Generate Python: Write a function to find the longest chain which can be formed from the given set of pairs.
Test cases:
assert max_chain_length([Pair(5, 24), Pair(15, 25),Pair(27, 40), Pair(50, 60)], 4) == 3
assert max_chain_length([Pair(1, 2), Pair(3, 4),Pair(5, 6), Pair(7, 8)], 4) == 4
assert max

In [5]:
mbpp_dataset

DatasetDict({
    train: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 374
    })
    test: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 90
    })
    prompt: Dataset({
        features: ['task_id', 'text', 'code', 'test_list', 'test_setup_code', 'challenge_test_list'],
        num_rows: 10
    })
})

In [6]:
mbpp_dataset['train'][0]

{'task_id': 601,
 'text': 'Write a function to find the longest chain which can be formed from the given set of pairs.',
 'code': 'class Pair(object): \r\n\tdef __init__(self, a, b): \r\n\t\tself.a = a \r\n\t\tself.b = b \r\ndef max_chain_length(arr, n): \r\n\tmax = 0\r\n\tmcl = [1 for i in range(n)] \r\n\tfor i in range(1, n): \r\n\t\tfor j in range(0, i): \r\n\t\t\tif (arr[i].a > arr[j].b and\r\n\t\t\t\tmcl[i] < mcl[j] + 1): \r\n\t\t\t\tmcl[i] = mcl[j] + 1\r\n\tfor i in range(n): \r\n\t\tif (max < mcl[i]): \r\n\t\t\tmax = mcl[i] \r\n\treturn max',
 'test_list': ['assert max_chain_length([Pair(5, 24), Pair(15, 25),Pair(27, 40), Pair(50, 60)], 4) == 3',
  'assert max_chain_length([Pair(1, 2), Pair(3, 4),Pair(5, 6), Pair(7, 8)], 4) == 4',
  'assert max_chain_length([Pair(19, 10), Pair(11, 12),Pair(13, 14), Pair(15, 16), Pair(31, 54)], 5) == 5'],
 'test_setup_code': '',
 'challenge_test_list': []}

In [7]:
# def process_mbpp_split(split):
#     """Convert a HuggingFace MBPP split into a list of input/output dicts."""
#     records = []
#     for ex in split:
#         records.append({
#             "input":     build_mbpp_prompt(ex),
#             "output":    ex["code"].strip(),
#             "task_id":   ex.get("task_id", ""),
#             "test_list": ex.get("test_list", []),   # kept for pass@1 evaluation
#         })
#     return records

# # Merge train + validation to maximise training examples
# mbpp_train = (
#     process_mbpp_split(mbpp_dataset["train"]) +
#     process_mbpp_split(mbpp_dataset["validation"])
# )
# mbpp_test = process_mbpp_split(mbpp_dataset["test"])

# print(f"\nMBPP → {len(mbpp_train):,} train (train+validation)  |  {len(mbpp_test):,} test")
# print("\nSample input prompt:")
# print(mbpp_train[0]["input"])
# print("\nSample target (Python code):")
# print(mbpp_train[0]["output"])

# write_jsonl(mbpp_train, MBPP_TRAIN_OUT)
# write_jsonl(mbpp_test,  MBPP_TEST_OUT)

In [8]:
# =============================================================================
# CELL 5 — SPIDER: Load from Drive & Preprocess
# =============================================================================
# Loads SPIDER question-SQL pairs directly from JSON files on Google Drive:
#   * train_spider.json  -> training set  (~7000 examples)
#   * train_others.json  -> test set      (treated as held-out evaluation set)
# Uses tables.json (also on Drive) for schema-enriched prompts.
#
# Prompt format:
#   Input : "Translate to SQL: {question}\nDatabase schema:\n{compact schema}"
#   Output: "{SQL query}"
#
# All three files must be uploaded to RAW_SPIDER_DIR on your Google Drive:
#   PROJECT_ROOT / data / raw / spider /
#
# Evaluation will use Exact Set Match (normalized SQL comparison).

# ── Step 1: Verify all required Drive files exist ─────────────────────────────
tables_json_path  = RAW_SPIDER_DIR / "tables.json"
train_spider_path = RAW_SPIDER_DIR / "train_spider.json"
train_others_path = RAW_SPIDER_DIR / "train_others.json"

for p in [tables_json_path, train_spider_path, train_others_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"Required file not found: {p}\n"
            f"Please upload it to your Google Drive at:\n"
            f"  {RAW_SPIDER_DIR}/"
        )

print("✓ All required SPIDER files found in Drive")

# ── Step 2: Load schemas from tables.json ─────────────────────────────────────
print(f"\nLoading schemas from {tables_json_path}...")
schema_lookup = load_spider_schemas(tables_json_path)
print(f"✓ Loaded schemas for {len(schema_lookup):,} databases")

# Preview one schema
sample_db = list(schema_lookup.keys())[0]
print(f"\nSample schema ({sample_db}):")
print(schema_lookup[sample_db])

# ── Step 3: Load raw examples from Drive JSON files ───────────────────────────
print(f"\nLoading train_spider.json from Drive...")
with open(train_spider_path, "r", encoding="utf-8") as f:
    raw_train = json.load(f)
print(f"  ✓ {len(raw_train):,} training examples loaded")

print(f"\nLoading train_others.json from Drive (used as test set)...")
with open(train_others_path, "r", encoding="utf-8") as f:
    raw_test = json.load(f)
print(f"  ✓ {len(raw_test):,} test examples loaded")

# ── Step 4: Format examples with schema-enriched prompts ─────────────────────
def process_spider_split(examples):
    """
    Convert a list of raw SPIDER dicts into schema-enriched input/output dicts.
    Each dict must have keys: 'question', 'query', 'db_id'.
    """
    records, skipped = [], 0
    for ex in examples:
        db_id = ex.get("db_id", "")
        if db_id not in schema_lookup:
            skipped += 1
            continue
        records.append({
            "input":    build_spider_prompt(ex, schema_lookup),
            "output":   ex["query"].strip(),
            "db_id":    db_id,
            "question": ex.get("question", ""),
        })
    if skipped:
        print(f"  Skipped {skipped} examples (db_id not found in tables.json)")
    return records

spider_train = process_spider_split(raw_train)
spider_test  = process_spider_split(raw_test)   # train_others.json = held-out test set

print(f"\nSPIDER → {len(spider_train):,} train  |  {len(spider_test):,} test")
print("\nSample input prompt:")
print(spider_train[0]["input"])
print("\nSample target (SQL):")
print(spider_train[0]["output"])

write_jsonl(spider_train, SPIDER_TRAIN_OUT)
write_jsonl(spider_test,  SPIDER_TEST_OUT)


✓ All required SPIDER files found in Drive

Loading schemas from /content/drive/MyDrive/FlanT5_Finetune/data/raw/spider/tables.json...
✓ Loaded schemas for 166 databases

Sample schema (perpetrator):
perpetrator: Perpetrator_ID (number), People_ID (number), Date (text), Year (number), Location (text), Country (text), Killed (number), Injured (number) | people: People_ID (number), Name (text), Height (number), Weight (number), Home Town (text)

Loading train_spider.json from Drive...
  ✓ 7,000 training examples loaded

Loading train_others.json from Drive (used as test set)...
  ✓ 1,659 test examples loaded

SPIDER → 7,000 train  |  1,659 test

Sample input prompt:
Translate to SQL: How many heads of the departments are older than 56 ?
Database schema:
department: Department_ID (number), Name (text), Creation (text), Ranking (number), Budget_in_Billions (number), Num_Employees (number) | head: head_ID (number), name (text), born_state (text), age (number) | management: department_ID (nu

In [9]:
print("\nSample input prompt:")
print(spider_train[10]["input"])
print("\nSample target (SQL):")
print(spider_train[10]["output"])


Sample input prompt:
Translate to SQL: How many acting statuses are there?
Database schema:
department: Department_ID (number), Name (text), Creation (text), Ranking (number), Budget_in_Billions (number), Num_Employees (number) | head: head_ID (number), name (text), born_state (text), age (number) | management: department_ID (number), head_ID (number), temporary_acting (text)

Sample target (SQL):
SELECT count(DISTINCT temporary_acting) FROM management


In [10]:
# =============================================================================
# CELL 6 — Dataset Statistics
# =============================================================================
# Tokenizes all examples and reports average / max token counts.
# Helps verify that max_input_length=512 covers most examples.

from transformers import AutoTokenizer

print(f"Loading tokenizer ({MODEL_NAME})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def print_token_stats(records, label):
    in_lens  = [len(tokenizer(r["input"],  truncation=False)["input_ids"]) for r in records]
    out_lens = [len(tokenizer(r["output"], truncation=False)["input_ids"]) for r in records]
    over_512 = sum(1 for l in in_lens if l > 512)
    print(f"\n{label}:")
    print(f"  Examples          : {len(records):,}")
    print(f"  Avg input tokens  : {sum(in_lens)  / len(in_lens):.1f}  (max: {max(in_lens)})")
    print(f"  Avg output tokens : {sum(out_lens) / len(out_lens):.1f}  (max: {max(out_lens)})")
    print(f"  Inputs > 512 tok  : {over_512} ({over_512 / len(records):.1%}) — will be truncated")

print_token_stats(mbpp_train,   "MBPP Train")
print_token_stats(mbpp_test,    "MBPP Test")
print_token_stats(spider_train, "SPIDER Train")
print_token_stats(spider_test,  "SPIDER Test")

print("\n✅ Preprocessing complete!")
print(f"   Files saved to: {PROCESSED_DIR}")
print("   → Proceed to 02_training.ipynb")


Loading tokenizer (google/flan-t5-base)...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (776 > 512). Running this sequence through the model will result in indexing errors



MBPP Train:
  Examples          : 464
  Avg input tokens  : 115.8  (max: 500)
  Avg output tokens : 73.5  (max: 297)
  Inputs > 512 tok  : 0 (0.0%) — will be truncated

MBPP Test:
  Examples          : 500
  Avg input tokens  : 121.0  (max: 2041)
  Avg output tokens : 74.2  (max: 443)
  Inputs > 512 tok  : 3 (0.6%) — will be truncated

SPIDER Train:
  Examples          : 7,000
  Avg input tokens  : 280.3  (max: 2503)
  Avg output tokens : 43.8  (max: 245)
  Inputs > 512 tok  : 686 (9.8%) — will be truncated

SPIDER Test:
  Examples          : 1,659
  Avg input tokens  : 245.5  (max: 518)
  Avg output tokens : 76.0  (max: 192)
  Inputs > 512 tok  : 3 (0.2%) — will be truncated

✅ Preprocessing complete!
   Files saved to: /content/drive/MyDrive/FlanT5_Finetune/data/processed
   → Proceed to 02_training.ipynb
